# PCAIR multigrid

The previous two notebooks established the core ingredients required for AIR methods:
1. The PMISR-DDC algorithm extracts a diagonally dominant submatrix $A_{ff}$.
2. GMRES polynomials provide effective approximate inverses for $A_{ff}$.

PCAIR combines these concepts to build a multigrid hierarchy by:
- Constructing a C/F splitting.
- Approximating ideal transfer operators (particularly the restriction operator) using approximations of $A_{ff}^{-1}$ computed with `PCPFLAREINV`.
- Forming a coarse grid operator.
- Repeating the process recursively on coarser levels.

The objective is to develop a solver or preconditioner that achieves algorithmic scalability for asymmetric problems—meaning the computational work scales linearly, $\mathcal{O}(N)$, with mesh refinement. Additionally, we aim for good parallel weak scaling, where the time to solution remains constant if the problem size and computational resources are increased proportionally.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

print("Ready.")

## 1. Challenges with standard methods for advection

Upwind discretizations of advection equations typically yield matrices that are asymmetric, non-normal, and lack strong diagonal dominance. These properties violate the fundamental assumptions underlying classical multigrid smoothers and transfer operators.

Common failure modes include:
- **Sweep-based methods** (e.g., Gauss-Seidel, ILU triangular solves) are highly sensitive to unknown ordering and difficult to parallelize, as they require sequential "sweeps" that follow the characteristic flow direction.
- **Standard AMG components** are optimized for SPD-like behavior, leading to severe convergence degradation on strongly asymmetric operators.
- **Block-local preconditioning** often loses algorithmic scalability as the number of subdomains increases.

AIR methods are specifically designed for this regime. They reformulate both the C/F splitting and the construction of transfer operators for asymmetric systems, discarding the traditional assumptions relied upon by SPD-focused multigrid methods.

## 2. The AIR multigrid framework

Given a linear system, we partition the unknowns into F-points and C-points:
$$
\begin{bmatrix} A_{ff} & A_{fc} \\ A_{cf} & A_{cc} \end{bmatrix}
\begin{bmatrix} x_f \\ x_c \end{bmatrix} =
\begin{bmatrix} b_f \\ b_c \end{bmatrix}.
$$

A useful theoretical reference is the exact block LDU factorization:
$$
A =
\underbrace{\begin{bmatrix} I & 0 \\ A_{cf}A_{ff}^{-1} & I \end{bmatrix}}_{L}
\underbrace{\begin{bmatrix} A_{ff} & 0 \\ 0 & S \end{bmatrix}}_{D}
\underbrace{\begin{bmatrix} I & A_{ff}^{-1}A_{fc} \\ 0 & I \end{bmatrix}}_{U},
\quad S = A_{cc} - A_{cf}A_{ff}^{-1}A_{fc}.
$$

The inverse of this factorization is given by:
$$
A^{-1} = U^{-1}D^{-1}L^{-1}
$$
with
$$
U^{-1} = \begin{bmatrix} I & -A_{ff}^{-1}A_{fc} \\ 0 & I \end{bmatrix},
\qquad
D^{-1} = \begin{bmatrix} A_{ff}^{-1} & 0 \\ 0 & S^{-1} \end{bmatrix},
\qquad
L^{-1} = \begin{bmatrix} I & 0 \\ -A_{cf}A_{ff}^{-1} & I \end{bmatrix}.
$$
The components of this inverse can be interpreted as a two-level method (or equivalently, block Gaussian elimination).

Conceptually, we apply $A_{ff}^{-1}$ to solve for the F-points. We then restrict the residual to the C-points and solve the coarse grid problem governed by the Schur complement $S^{-1}$. Finally, we prolongate the coarse correction back to the fine grid, yielding an exact solution across all unknowns.

This exact factorization motivates the definition of the ideal AIR transfer operators:
$$
W = -A_{ff}^{-1}A_{fc}, \qquad Z = -A_{cf}A_{ff}^{-1},
$$
where $P = \begin{bmatrix} W \\ I \end{bmatrix}$ is the ideal prolongator and $R = \begin{bmatrix} Z & I \end{bmatrix}$ is the ideal restrictor.

While practical AIR multigrid does not apply the exact block-LDU inverse, it mimics this structure. Instead of computing the exact inverse of $A_{ff}$, we use an approximation to form an approximate ideal restrictor (and/or prolongator).

The key to a scalable AIR method lies in selecting F-points such that $A_{ff}^{-1}$ can be approximated both cheaply and accurately on every level. A higher-quality approximation of $A_{ff}^{-1}$ leads to better transfer operators ($W$, $Z$) and more effective F-point smoothing, ultimately driving robust convergence.

In [ ]:
def build_2d_adv_diff(N, eps=1e-4):
    """2D advection-diffusion: -eps*Lap(u) + u_x = 1 with 5-point FD."""
    h = 1.0 / (N + 1)
    n = N * N
    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(5)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    for row in range(rstart, rend):
        i, j = divmod(row, N)
        diag = 4.0 * eps / h**2 + 1.0 / h
        A.setValue(row, row, diag)
        if j > 0:     A.setValue(row, row - 1, -eps / h**2 - 1.0 / h)
        if j < N - 1: A.setValue(row, row + 1, -eps / h**2)
        if i > 0:     A.setValue(row, row - N, -eps / h**2)
        if i < N - 1: A.setValue(row, row + N, -eps / h**2)
    A.assemblyBegin(); A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    return A, b

print('Matrix builders ready.')

## 3. The AIRG variant

The AIRG variant specifically employs the GMRES polynomials discussed in previous notebooks to approximate $A_{ff}^{-1}$. As shown above, the ideal restrictor component is $Z = -A_{cf}A_{ff}^{-1}$.

This motivates the use of assembled approximations for the GMRES polynomials: it allows us to compute $Z$ efficiently via a sparse matrix-matrix product, while retaining the assembled polynomial matrix to serve as a powerful F-point smoother.

Alternatively, a matrix-free option can be enabled. In this mode, the assembled approximation is built temporarily to compute $Z$ and then discarded to conserve memory. The F-point smoothing is subsequently performed matrix-free by reusing the polynomial coefficients. This approach can be highly efficient on GPU architectures, which excel at executing repeated matrix-vector products.

## 4. Performance on a 2D advection operator

We now demonstrate the effectiveness of PCAIR on asymmetric systems. By default, PCAIR utilizes:
1. PMISR-DDC for the C/F splitting.
2. The AIRG variant for hierarchy construction.
3. Assembled 6th-order GMRES polynomials with 1st-order fixed sparsity to approximate $A_{ff}^{-1}$.

We evaluate performance using a 2D upwind advection operator, comparing PCAIR against an ILU preconditioner and unpreconditioned GMRES. The unknowns are deliberately not ordered to follow the advection flow. While AIRG and standard GMRES are insensitive to ordering, ILU is highly dependent on it. Although an optimal ordering could allow ILU to converge in a single iteration, the required forward/backward substitutions severely limit parallel scalability.

In [ ]:
def count_iterations(A, b, pc_type, pc_options=None, max_it=500, rtol=1e-10):
    """Run GMRES and return iteration count (unpreconditioned norm).
    Pass PETSc option names exactly as you want them after the `pcair_` KSP prefix,
    e.g. `pc_air_print_stats_timings`, `pc_air_z_type`, `ksp_view`."""
    pc_options = pc_options or {}
    opts = PETSc.Options()
    prefix = 'pcair_'

    set_keys = []
    for key, value in pc_options.items():
        full_key = prefix + key

        is_view_flag = key.endswith('_view')
        if is_view_flag and str(value).strip().lower() in ('1', 'true', 'yes', 'on'):
            opts[full_key] = ''
        elif value is True or value is None:
            opts[full_key] = ''
        else:
            opts[full_key] = str(value)

        set_keys.append(full_key)

    ksp = PETSc.KSP().create()
    ksp.setOptionsPrefix(prefix)
    ksp.setOperators(A)
    ksp.setType('gmres')
    ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    ksp.getPC().setType(pc_type)
    ksp.setFromOptions()

    x = A.createVecRight()
    x.set(0.0)
    ksp.solve(b, x)

    iters = ksp.getIterationNumber()
    reason = ksp.getConvergedReason()
    if reason < 0:
        iters = max_it

    for key in set_keys:
        try:
            del opts[key]
        except Exception:
            pass

    return iters

# Small practical experiment: strongly asymmetric 1D upwind sequence
grid_sizes = [50, 100, 200, 400]
iters_air = []
iters_ilu = []
iters_none = []

for N in grid_sizes:
    A, b = build_2d_adv_diff(N)

    n_air = count_iterations(
        A, b, 'air',
        max_it=500, rtol=1e-10,
    )

    n_ilu = count_iterations(A, b, 'ilu', max_it=500, rtol=1e-10)
    n_none = count_iterations(A, b, 'none', max_it=1000, rtol=1e-10)

    iters_air.append(n_air)
    iters_ilu.append(n_ilu)
    iters_none.append(n_none)

    A.destroy(); b.destroy()
    print(f'N={N:4d}: PCAIR={n_air:4d}  ILU={n_ilu:4d}  None={n_none:4d}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(grid_sizes, iters_none, 'k--o', label='No preconditioner')
ax.loglog(grid_sizes, iters_ilu, 'g-^', label='ILU(0)')
ax.loglog(grid_sizes, iters_air, 'r-s', label='PCAIR')
ax.set_xlabel('Problem size N (1D upwind, N unknowns)')
ax.set_ylabel('GMRES iterations to rtol=1e-10 (unpreconditioned norm)')
ax.set_title('Asymmetric advection: iteration growth comparison')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

The results show that AIRG performs well, maintaining a nearly flat iteration count as the problem size increases. Notably, this is achieved using the default parameters; while tuning is often necessary for optimal performance at larger scales, the defaults provide an excellent baseline.

## 5. Hierarchy statistics and parameter variations

By providing the `-pc_air_print_stats_timings` option, we can inspect the hierarchy construction, view cumulative timing results, and analyze statistics of the resulting multigrid setup.

In [ ]:
# Run one PCAIR solve with hierarchy/stat printing enabled
A, b = build_2d_adv_diff(100)

pcair_stats_options = {
    'pc_air_print_stats_timings': '1',
}

n_air_stats = count_iterations(
    A, b, 'air',
    pc_options=pcair_stats_options,
    max_it=500,
    rtol=1e-10,
 )

print(f"PCAIR (with hierarchy stats) iterations = {n_air_stats}")

A.destroy()
b.destroy()

The output indicates that a 12-level multigrid hierarchy was formed, with summary statistics provided at the end. The cycle complexity and storage complexity are particularly important metrics.

The **cycle complexity** represents the equivalent number of fine-grid matrix-vector products required for one AIR iteration. Multiplying this by the total iteration count provides a rough estimate of the total computational work. While this metric may not perfectly predict wall-clock time in parallel or GPU environments, it is a valuable proxy for algorithmic cost.

The **storage complexity** indicates the total memory footprint of the hierarchy, expressed as a multiple of the memory required to store the fine-grid matrix.

Parameters such as drop tolerances, coarsening thresholds, and matrix-free smoothing significantly impact these statistics. For instance, enabling matrix-free smoothing increases the cycle complexity (due to the repeated matrix-vector products required for F-point smoothing) but substantially reduces the storage complexity.

In [ ]:
# Run one PCAIR solve with hierarchy/stat printing enabled
A, b = build_2d_adv_diff(100)

pcair_stats_options = {
    'pc_air_print_stats_timings': '1',
    'pc_air_matrix_free_polys': '1',
}

n_air_stats = count_iterations(
    A, b, 'air',
    pc_options=pcair_stats_options,
    max_it=500,
    rtol=1e-10,
 )

print(f"PCAIR (with hierarchy stats) iterations = {n_air_stats}")

A.destroy()
b.destroy()

## 6. AIR variants

PCAIR supports several AIR variants, which differ primarily in their approach to transfer operator approximation and smoother selection:
- **AIRG**: Uses GMRES polynomials to approximate $A_{ff}^{-1}$.
- **nAIR**: Uses Neumann polynomials to approximate $A_{ff}^{-1}$.
- **lAIR**: Approximates the restrictor $Z$ directly, rather than via $A_{ff}^{-1}$.

For example, we can enable lAIR transfer operators combined with GMRES polynomial smoothing by specifying `-pc_air_z_type lair`. We can also inspect the full set of options used during the solve by adding the `ksp_view` flag.

In [ ]:
# Run one PCAIR solve with hierarchy/stat printing enabled
A, b = build_2d_adv_diff(100)

pcair_stats_options = {
    'pc_air_print_stats_timings': '1',
    'pc_air_z_type': 'lair',
    'ksp_view': '',
}

n_air_stats = count_iterations(
    A, b, 'air',
    pc_options=pcair_stats_options,
    max_it=500,
    rtol=1e-10,
 )

print(f"PCAIR (with hierarchy stats) iterations = {n_air_stats}")

A.destroy()
b.destroy()

## Summary

- Standard iterative methods and classical multigrid often struggle or fail on strongly asymmetric, non-normal systems.
- AIR methods reformulate the multigrid approach around F/C reduction and the approximation of ideal transfer operators.
- The AIRG variant provides robust, scalable solutions for challenging advection-dominated problems.